<a href="https://colab.research.google.com/github/evinracher/3010090-ontological-engineering/blob/main/week6/01_ReActAgents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentes ReAct con LangChain, LangGraph y Groq

## Objetivo
Este cuaderno muestra la idea detrás de **ReAct**, una arquitectura general de agentes.

1. **actuar**: permitir que el modelo invoque herramientas específicas.
2. **observar**: devolver al modelo la salida producida por la herramienta.
3. **razonar**: permitir que el modelo razone sobre esa salida para decidir el siguiente paso (por ejemplo, llamar otra herramienta o responder directamente).


In [ ]:
# Instalación de dependencias para Google Colab
!pip -q install -U langchain langchain-community langchain-core langgraph langchain-groq tavily-python wikipedia arxiv python-dotenv


**BUSCADOR WEB TAVILY**

Es uno de los motores de búsqueda más usados porque está optimizado para consultas de LLMs.

Proporciona una API de búsqueda web diseñada para sistemas de inteligencia artificial.

Requiere una API KEY. Es neccesario crear una cuenta. Hay una version gratuita que se puede usar de manera limitada.

Página Web:

https://app.tavily.com/home

En LangChain se usa principalmente en:

* Agentes ReAct
* Agentes con herramientas
* RAG con búsqueda web



In [ ]:
# Cargar claves desde Google Colab Secrets o variables de entorno
import os

try:
    from google.colab import userdata
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "") # TAVILY es
    os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

print("Claves cargadas correctamente.")


## 1) Preparacion de Herramientas (tools)

Se preparan varias herramientas:

* ArxivQueryRun: Herramienta (tool) de LangChain que permite a un agente consultar directamente el repositorio científico arXiv (https://arxiv.org/) para buscar artículos académicos
* WikipediaQueryRun
* TavilySearchResults
* funciones matemáticas como add, divide, multiply

In [ ]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [ ]:
### Herramienta (Tool) ArxivQueryRun
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

In [ ]:
arxiv.invoke("Attention Is All You Need")


In [ ]:
### Herramienta (Tool) WikipediaQueryRun
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=500)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

In [ ]:
wiki.invoke("¿Qué es el aprendizaje automático?")


In [ ]:
### Funciones personalizadas
def multiply(a: int, b: int) -> int:
    """Multiplica a y b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a * b

# Esta función se convertirá en una herramienta
def add(a: int, b: int) -> int:
    """Suma a y b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a y b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a / b

tools = [arxiv, wiki, add, multiply, divide]


In [ ]:
### Herramienta de búsqueda Tavily
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()


In [ ]:
tavily.invoke("Dame las noticias recientes de IA del 3 de marzo de 2025")


In [ ]:
### Combinar todas las herramientas en la lista

tools = [arxiv, wiki, tavily, add, divide, multiply]


Combianar las herramientas es importante porque en ReAct el modelo no responde directamente siempre; primero puede decidir qué herramienta usar.

## 2) Habilitar el LLM para llamar herramientas

`bind_tools(tools)` convierte el modelo en un LLM que ya puede:

* Decidir usar una herramienta
* Seleccionar cuál
* Generar los argumentos de entrada

**OBSERVACION:** Esto corresponde a la parte Act de ReAct.

In [ ]:
## Inicializar el modelo LLM

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

llm_with_tools = llm.bind_tools(tools)


# 3) Verificar que el modelo produce `tool_calls`

Se pretende demostrar que el modelo no solo responde con texto, sino que puede emitir una intención estructurada de usar una herramienta.

En otras palabras, el LLM hace algo equivalente a:
* “Necesito buscar en la web”
* “Voy a usar Tavily”
* “Con esta consulta”

**OBSERVACION:** Este comportamiento es exactamente el patrón ReAct.

In [ ]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage
llm_with_tools.invoke([HumanMessage(content="¿Cuáles son las noticias recientes de IA?")])


In [ ]:
llm_with_tools.invoke([HumanMessage(content="¿Cuáles son las noticias recientes de IA?")]).tool_calls


# 4) El estado del agente guarda la conversación y las observaciones

Se define un estado: State(TypedDict).

Esto significa que el sistema conserva una secuencia de mensajes.
Ahí se van acumulando:

* la pregunta del usuario
* la respuesta del LLM
* la llamada a herramienta
* la salida de la herramienta
* la respuesta final

Ese historial es fundamental en **ReAct**, porque el modelo necesita ver la observación anterior para razonar el siguiente paso.

In [ ]:
## Esquema del estado
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


# 5) Crear el Workflow

a. Se crea un nodo de razonamiento del LLM

b. Se crea un nodo separado para ejecutar herramientas

c. Se crea El grafo que implementa el bucle **ReAct**. Este patrón significa:

1.   Entra la pregunta
2.   El LLM razona
3.   si pidió herramienta, va al nodo `tools`
4.   la herramienta devuelve una observación
5.   se regresa al LLM
6.   el LLM vuelve a razonar con esa observación
7.   si ya no necesita herramientas, termina


Esto no es una cadena lineal común.
Es un ciclo de decisión y acción, que es precisamente lo que define a **ReAct**.

Pro último, la función `tools_condition` revisa el mensaje más reciente del asistente. Esto equivale a esta lógica:

* “todavía necesito actuar”

o

* “ya tengo suficiente información, puedo responder”

Este mecanismo de control es característico de una arquitectura **ReAct**.


In [ ]:
### Chatbot completo con LangGraph
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

### Definición del nodo
def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# Construir el grafo
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # Si el mensaje más reciente del asistente es una llamada a herramienta, tools_condition redirige a tools
    # Si el mensaje más reciente del asistente NO es una llamada a herramienta, tools_condition redirige a END
    tools_condition,
)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

# Visualizar
display(Image(graph.get_graph().draw_mermaid_png()))


# 6) Ejecución final que muestra múltiples pasos

El agente puede:

* usar Tavily para noticias
* usar add
* usar multiply
* combinar observaciones
* producir una respuesta final

Esto demuestra que el sistema descompone la tarea en pasos, usa herramientas intermedias y luego integra resultados.

Este comportamiento multi-step es uno de los rasgos más claros de **ReAct**.

In [ ]:
messages = graph.invoke({"messages": HumanMessage(content="Dame las 10 noticias recientes de IA del 3 de marzo de 2025, suma 5 más 5 y luego multiplícalo por 10")})
for m in messages["messages"]:
    m.pretty_print()
